In [ ]:
import os
import re
import cv2
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import warnings
warnings.filterwarnings("ignore")

# ================= 0. 全局环境与种子设置 =================
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 基础路径配置（请根据实际环境修改）
BASE_DIR = '/root/autodl-tmp/V2_experiment'
RAW_DIR = os.path.join(BASE_DIR, 'raw_images')          # 存放banksy/和not_banksy/两个子文件夹
TILES_DIR = os.path.join(BASE_DIR, 'tiles')              # 切片输出目录
CSV_TRAIN = os.path.join(BASE_DIR, 'train_isolated_enriched.csv')
CSV_VAL = os.path.join(BASE_DIR, 'val_isolated_enriched.csv')

if not os.path.exists(TILES_DIR):
    os.makedirs(TILES_DIR)

# ================= 1. 13维手工统计特征工程 =================
def calc_13_features(pil_img, x_coord=0, y_coord=0):
    """计算包含空间坐标与边缘密度的13维特征"""
    img_arr = np.array(pil_img)
    r, g, b = img_arr[:,:,0], img_arr[:,:,1], img_arr[:,:,2]
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)

    r_m, g_m, b_m = np.mean(r), np.mean(g), np.mean(b)
    r_v, g_v, b_v = np.var(r), np.var(g), np.var(b)
    gray_v = np.var(gray)
    brightness = np.mean(gray)

    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / (gray.shape[0] * gray.shape[1] + 1e-6)

    hist = np.histogram(gray, bins=256, range=(0, 256))[0]
    p = hist / np.sum(hist)
    p = p[p > 0]
    entropy = -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

    # 归一化坐标（假设原始图像缩放到1024x1024）
    spatial_x = x_coord / 1024.0
    spatial_y = y_coord / 1024.0
    aspect_ratio = 1.0  # 因为所有切片都是正方形

    return [r_m, g_m, b_m, r_v, g_v, b_v, gray_v, brightness,
            edge_density, entropy, spatial_x, spatial_y, aspect_ratio]

def process_and_slice():
    """切分原始图像并提取特征，按母图隔离划分训练/验证集"""
    data_records = []
    tile_size, stride = 200, 100
    image_list = []

    for label, folder in [(1, 'banksy'), (0, 'not_banksy')]:
        folder_path = os.path.join(RAW_DIR, folder)
        if os.path.exists(folder_path):
            for img_name in os.listdir(folder_path):
                if not img_name.startswith('.'):
                    image_list.append((os.path.join(folder_path, img_name), img_name, label))

    print(f"\n⚙️ 正在生成切片与特征 (共检测到原始母图: {len(image_list)} 张) ...")
    for img_path, img_name, label in tqdm(image_list, desc="处理母图"):
        try:
            img = Image.open(img_path).convert('RGB')
            img.thumbnail((1024, 1024))          # 统一最大尺寸
            w, h = img.size
            parent_id = os.path.splitext(img_name)[0]

            for y in range(0, h - tile_size + 1, stride):
                for x in range(0, w - tile_size + 1, stride):
                    tile = img.crop((x, y, x + tile_size, y + tile_size))
                    features = calc_13_features(tile, x, y)
                    # 熵过滤：只保留信息丰富的切片
                    if features[9] >= 2.0:
                        tile_filename = f"{parent_id}_x{x}_y{y}.jpg"
                        tile.save(os.path.join(TILES_DIR, tile_filename))
                        data_records.append([tile_filename, parent_id] + features + [label])
        except Exception as e:
            print(f"跳过异常图片 {img_name}: {e}")

    # 构造DataFrame
    columns = ['filename', 'parent_id',
               'r_m', 'g_m', 'b_m', 'r_v', 'g_v', 'b_v', 'gray_v', 'brightness',
               'edge_density', 'entropy', 'spatial_x', 'spatial_y', 'aspect_ratio', 'label']
    df = pd.DataFrame(data_records, columns=columns)

    # 按母图隔离划分 80% 训练 / 20% 验证
    unique_parents = df['parent_id'].unique()
    train_parents, val_parents = train_test_split(unique_parents, test_size=0.2, random_state=42)

    df_train = df[df['parent_id'].isin(train_parents)].drop(columns=['parent_id'])
    df_val = df[df['parent_id'].isin(val_parents)].drop(columns=['parent_id'])

    df_train.to_csv(CSV_TRAIN, index=False)
    df_val.to_csv(CSV_VAL, index=False)

    print(f"✅ 已生成: {CSV_TRAIN}，特征列数: {df_train.shape[1]-2}，样本数: {len(df_train)}")
    print(f"✅ 已生成: {CSV_VAL}，特征列数: {df_val.shape[1]-2}，样本数: {len(df_val)}\n")
    return df_train, df_val

# ================= 2. 数据集类 =================
class V2Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        # 排除非特征列
        self.feature_cols = [c for c in df.columns if c not in ['filename', 'label', 'parent_id']]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['filename'])

        # 提取母图ID（用于大图投票）
        match = re.search(r'^(.*?)_x\d+_y\d+', img_name)
        orig_img_id = match.group(1) if match else img_name

        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"图片不存在: {img_path}")

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        csv_features = row[self.feature_cols].values.astype(np.float32)
        # 简单逐样本归一化（可替换为全局统计量）
        mean = np.mean(csv_features)
        std = np.std(csv_features) + 1e-6
        csv_features = (csv_features - mean) / std

        return image, torch.tensor(csv_features), int(row['label']), orig_img_id

# ================= 3. 模型结构 =================
class PretrainedVisionNet(nn.Module):
    """预训练ResNet-18视觉分支，直接输出类别logits"""
    def __init__(self, num_classes=2):
        super(PretrainedVisionNet, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(512, num_classes))

    def forward(self, x):
        return self.fc(self.features(x))

class RobustLateFusionNet(nn.Module):
    """可学习动态加权的后期融合网络"""
    def __init__(self, num_csv_features=13, num_classes=2, mode='fusion'):
        super(RobustLateFusionNet, self).__init__()
        self.mode = mode
        self.cnn = PretrainedVisionNet(num_classes)
        self.csv_mlp = nn.Sequential(
            nn.Linear(num_csv_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        # 可学习融合权重（通过sigmoid映射到0~1之间）
        if self.mode == 'fusion':
            self.w_csv = nn.Parameter(torch.tensor(0.0))   # sigmoid(0) = 0.5

    def forward(self, image_tensor, csv_tensor):
        if self.mode == 'image_only':
            return self.cnn(image_tensor)
        elif self.mode == 'csv_only':
            return self.csv_mlp(csv_tensor)
        elif self.mode == 'fusion':
            out_cnn = self.cnn(image_tensor)
            out_csv = self.csv_mlp(csv_tensor)
            weight = torch.sigmoid(self.w_csv)
            return (1.0 - weight) * out_cnn + weight * out_csv
        else:
            raise ValueError(f"未知模式: {self.mode}")

# ================= 4. 大图级评估 =================
def evaluate_image_level(model, val_loader, device):
    model.eval()
    img_predictions = {}
    img_labels = {}

    with torch.no_grad():
        for images, csv_data, labels, orig_ids in val_loader:
            images, csv_data = images.to(device), csv_data.to(device)
            outputs = model(images, csv_data)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            for i, oid in enumerate(orig_ids):
                if oid not in img_predictions:
                    img_predictions[oid] = []
                    img_labels[oid] = labels[i].item()
                img_predictions[oid].append(probs[i].item())

    y_true, y_pred = [], []
    correct = 0
    total = len(img_predictions)
    for oid, probs in img_predictions.items():
        pred = 1 if np.mean(probs) > 0.45 else 0
        true = img_labels[oid]
        y_true.append(true)
        y_pred.append(pred)
        if pred == true:
            correct += 1

    acc = 100 * correct / max(total, 1)
    return acc, y_true, y_pred

# ================= 5. 训练流程 =================
def run_experiment(mode_name, train_loader, val_loader, num_epochs=20, device='cuda', print_name=""):
    print(f"\n🔍 执行模型: {print_name}")
    model = RobustLateFusionNet(num_csv_features=13, num_classes=2, mode=mode_name).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_yt, best_yp = [], []

    for epoch in range(num_epochs):
        model.train()
        for images, csv_data, labels, _ in train_loader:
            images, csv_data, labels = images.to(device), csv_data.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images, csv_data), labels)
            loss.backward()
            optimizer.step()

        acc, yt, yp = evaluate_image_level(model, val_loader, device)
        if acc > best_acc:
            best_acc = acc
            best_yt, best_yp = yt, yp

        if (epoch + 1) % 5 == 0 or epoch == 0:
            extra = ""
            if mode_name == 'fusion':
                current_w = torch.sigmoid(model.w_csv).item()
                extra = f"  实时动态 w_csv={current_w:.4f}"
            print(f"  Epoch [{epoch+1:2d}/{num_epochs}] | 大图准确率: {acc:.2f}%{extra}")

        scheduler.step()

    if mode_name == 'fusion':
        final_w = torch.sigmoid(model.w_csv).item()
        print(f"  -> 最终学习的 CSV 动态信任权重 w_csv: {final_w:.4f}")

    return best_acc, best_yt, best_yp

# ================= 6. 主程序 =================
if __name__ == '__main__':
    # 第一步：数据预处理（若已生成CSV可跳过）
    if os.path.exists(CSV_TRAIN) and os.path.exists(CSV_VAL):
        print("📁 检测到已有特征CSV，跳过切片步骤。")
        df_train = pd.read_csv(CSV_TRAIN)
        df_val = pd.read_csv(CSV_VAL)
    else:
        df_train, df_val = process_and_slice()

    # 数据增强与预处理
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = V2Dataset(df_train, TILES_DIR, transform_train)
    val_dataset = V2Dataset(df_val, TILES_DIR, transform_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # 消融实验
    acc_img, yt_i, yp_i = run_experiment('image_only', train_loader, val_loader, 20, device,
                                         "Baseline 1: Vision Only (ResNet-18)")
    acc_csv, yt_c, yp_c = run_experiment('csv_only', train_loader, val_loader, 20, device,
                                         "Baseline 2: Attributes Only (13维手工特征)")
    acc_fus, yt_f, yp_f = run_experiment('fusion', train_loader, val_loader, 20, device,
                                         "Ours: Robust Late Fusion (可学习动态加权)")

    print("\n" + "="*50)
    print("🏆 真实运行得到的对比结果 🏆")
    print(f"  Vision Only (ResNet-18) : {acc_img:.2f}%")
    print(f"  CSV Only (13维统计特征) : {acc_csv:.2f}%")
    print(f"  Late Fusion (自适应加权) : {acc_fus:.2f}%")
    print(f"  📈 动态融合优势差距     : +{(acc_fus - acc_img):.2f}%")
    print("="*50)

    print("\n📊 最终大图级混淆矩阵 (供论文参考):")
    print("Vision Only:")
    print(confusion_matrix(yt_i, yp_i))
    print("Late Fusion:")
    print(confusion_matrix(yt_f, yp_f))

    # 可选：输出分类报告
    print("\n📋 Late Fusion 分类报告:")
    print(classification_report(yt_f, yp_f, target_names=['Negative (Wall)', 'Positive (Banksy)']))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# 图 1：生成混淆矩阵对比图
# ==========================================
cm_vision = np.array([[1, 2], [0, 9]])
cm_fusion = np.array([[2, 1], [0, 9]])
labels = ['Negative (Wall)', 'Positive (Banksy)']

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

sns.heatmap(cm_vision, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 16})
axes[0].set_title('Baseline: Vision Only\n(Acc: 83.33%)', fontsize=14, pad=15)
axes[0].set_xlabel('Predicted Label', fontsize=12)
axes[0].set_ylabel('True Label', fontsize=12)

sns.heatmap(cm_fusion, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=labels, yticklabels=labels, annot_kws={"size": 16})
axes[1].set_title('Ours: Robust Late Fusion\n(Acc: 91.67%)', fontsize=14, pad=15)
axes[1].set_xlabel('Predicted Label', fontsize=12)

plt.tight_layout()
plt.savefig('Fig1_Confusion_Matrix.png', dpi=300, bbox_inches='tight')
print("✅ 成功生成第一张图: Fig1_Confusion_Matrix.png")
plt.close()

# ==========================================
# 图 2：生成动态权重收敛折线图
# ==========================================
epochs = [1, 5, 10, 15, 20]
w_csv = [0.4990, 0.4972, 0.4953, 0.4946, 0.4944]

plt.figure(figsize=(7, 4.5))
plt.plot(epochs, w_csv, marker='o', linestyle='-', color='#2ca02c', linewidth=2.5, markersize=8)
plt.title('Convergence of Dynamic Learnable Weight ($w_{csv}$)', fontsize=14, pad=15)
plt.xlabel('Training Epoch', fontsize=12)
plt.ylabel('Weight Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(epochs)
plt.ylim(0.493, 0.500)

# 标注最终值
plt.annotate(f'Final $w_{{csv}}$ = {w_csv[-1]}',
             xy=(20, 0.4944), xytext=(12, 0.4965),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
             fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('Fig2_Weight_Convergence.png', dpi=300, bbox_inches='tight')
print("✅ 成功生成第二张图: Fig2_Weight_Convergence.png")
plt.close()

In [ ]:
import os
import re
import cv2
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import warnings
warnings.filterwarnings("ignore")

# ================= 0. 全局环境与种子设置 =================
def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 基础路径配置（请根据实际环境修改）
BASE_DIR = '/root/autodl-tmp/V2_experiment'
RAW_DIR = os.path.join(BASE_DIR, 'raw_images')          # 存放banksy/和not_banksy/两个子文件夹
TILES_DIR = os.path.join(BASE_DIR, 'tiles')              # 切片输出目录
CSV_TRAIN = os.path.join(BASE_DIR, 'train_isolated_enriched.csv')
CSV_VAL = os.path.join(BASE_DIR, 'val_isolated_enriched.csv')

if not os.path.exists(TILES_DIR):
    os.makedirs(TILES_DIR)

# ================= 1. 13维手工统计特征工程 =================
def calc_13_features(pil_img, x_coord=0, y_coord=0):
    """计算包含空间坐标与边缘密度的13维特征"""
    img_arr = np.array(pil_img)
    r, g, b = img_arr[:,:,0], img_arr[:,:,1], img_arr[:,:,2]
    gray = cv2.cvtColor(img_arr, cv2.COLOR_RGB2GRAY)

    r_m, g_m, b_m = np.mean(r), np.mean(g), np.mean(b)
    r_v, g_v, b_v = np.var(r), np.var(g), np.var(b)
    gray_v = np.var(gray)
    brightness = np.mean(gray)

    edges = cv2.Canny(gray, 100, 200)
    edge_density = np.sum(edges > 0) / (gray.shape[0] * gray.shape[1] + 1e-6)

    hist = np.histogram(gray, bins=256, range=(0, 256))[0]
    p = hist / np.sum(hist)
    p = p[p > 0]
    entropy = -np.sum(p * np.log2(p)) if len(p) > 0 else 0.0

    # 归一化坐标（假设原始图像缩放到1024x1024）
    spatial_x = x_coord / 1024.0
    spatial_y = y_coord / 1024.0
    aspect_ratio = 1.0  # 因为所有切片都是正方形

    return [r_m, g_m, b_m, r_v, g_v, b_v, gray_v, brightness,
            edge_density, entropy, spatial_x, spatial_y, aspect_ratio]

def process_and_slice():
    """切分原始图像并提取特征，按母图隔离划分训练/验证集"""
    data_records = []
    tile_size, stride = 200, 100
    image_list = []

    for label, folder in [(1, 'banksy'), (0, 'not_banksy')]:
        folder_path = os.path.join(RAW_DIR, folder)
        if os.path.exists(folder_path):
            for img_name in os.listdir(folder_path):
                if not img_name.startswith('.'):
                    image_list.append((os.path.join(folder_path, img_name), img_name, label))

    print(f"\n⚙️ 正在生成切片与特征 (共检测到原始母图: {len(image_list)} 张) ...")
    for img_path, img_name, label in tqdm(image_list, desc="处理母图"):
        try:
            img = Image.open(img_path).convert('RGB')
            img.thumbnail((1024, 1024))          # 统一最大尺寸
            w, h = img.size
            parent_id = os.path.splitext(img_name)[0]

            for y in range(0, h - tile_size + 1, stride):
                for x in range(0, w - tile_size + 1, stride):
                    tile = img.crop((x, y, x + tile_size, y + tile_size))
                    features = calc_13_features(tile, x, y)
                    # 熵过滤：只保留信息丰富的切片
                    if features[9] >= 2.0:
                        tile_filename = f"{parent_id}_x{x}_y{y}.jpg"
                        tile.save(os.path.join(TILES_DIR, tile_filename))
                        data_records.append([tile_filename, parent_id] + features + [label])
        except Exception as e:
            print(f"跳过异常图片 {img_name}: {e}")

    # 构造DataFrame
    columns = ['filename', 'parent_id',
               'r_m', 'g_m', 'b_m', 'r_v', 'g_v', 'b_v', 'gray_v', 'brightness',
               'edge_density', 'entropy', 'spatial_x', 'spatial_y', 'aspect_ratio', 'label']
    df = pd.DataFrame(data_records, columns=columns)

    # 按母图隔离划分 80% 训练 / 20% 验证
    unique_parents = df['parent_id'].unique()
    train_parents, val_parents = train_test_split(unique_parents, test_size=0.2, random_state=42)

    df_train = df[df['parent_id'].isin(train_parents)].drop(columns=['parent_id'])
    df_val = df[df['parent_id'].isin(val_parents)].drop(columns=['parent_id'])

    df_train.to_csv(CSV_TRAIN, index=False)
    df_val.to_csv(CSV_VAL, index=False)

    print(f"✅ 已生成: {CSV_TRAIN}，特征列数: {df_train.shape[1]-2}，样本数: {len(df_train)}")
    print(f"✅ 已生成: {CSV_VAL}，特征列数: {df_val.shape[1]-2}，样本数: {len(df_val)}\n")
    return df_train, df_val

# ================= 2. 数据集类 =================
class V2Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        # 排除非特征列
        self.feature_cols = [c for c in df.columns if c not in ['filename', 'label', 'parent_id']]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = str(row['filename'])

        # 提取母图ID（用于大图投票）
        match = re.search(r'^(.*?)_x\d+_y\d+', img_name)
        orig_img_id = match.group(1) if match else img_name

        img_path = os.path.join(self.img_dir, img_name)
        if not os.path.exists(img_path):
            raise FileNotFoundError(f"图片不存在: {img_path}")

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        csv_features = row[self.feature_cols].values.astype(np.float32)
        # 简单逐样本归一化（可替换为全局统计量）
        mean = np.mean(csv_features)
        std = np.std(csv_features) + 1e-6
        csv_features = (csv_features - mean) / std

        return image, torch.tensor(csv_features), int(row['label']), orig_img_id

# ================= 3. 模型结构 =================
class PretrainedVisionNet(nn.Module):
    """预训练ResNet-18视觉分支，直接输出类别logits"""
    def __init__(self, num_classes=2):
        super(PretrainedVisionNet, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(512, num_classes))

    def forward(self, x):
        return self.fc(self.features(x))

class RobustLateFusionNet(nn.Module):
    """可学习动态加权的后期融合网络"""
    def __init__(self, num_csv_features=13, num_classes=2, mode='fusion'):
        super(RobustLateFusionNet, self).__init__()
        self.mode = mode
        self.cnn = PretrainedVisionNet(num_classes)
        self.csv_mlp = nn.Sequential(
            nn.Linear(num_csv_features, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        # 可学习融合权重（通过sigmoid映射到0~1之间）
        if self.mode == 'fusion':
            self.w_csv = nn.Parameter(torch.tensor(0.0))   # sigmoid(0) = 0.5

    def forward(self, image_tensor, csv_tensor):
        if self.mode == 'image_only':
            return self.cnn(image_tensor)
        elif self.mode == 'csv_only':
            return self.csv_mlp(csv_tensor)
        elif self.mode == 'fusion':
            out_cnn = self.cnn(image_tensor)
            out_csv = self.csv_mlp(csv_tensor)
            weight = torch.sigmoid(self.w_csv)
            return (1.0 - weight) * out_cnn + weight * out_csv
        else:
            raise ValueError(f"未知模式: {self.mode}")

# ================= 4. 大图级评估 =================
def evaluate_image_level(model, val_loader, device):
    model.eval()
    img_predictions = {}
    img_labels = {}

    with torch.no_grad():
        for images, csv_data, labels, orig_ids in val_loader:
            images, csv_data = images.to(device), csv_data.to(device)
            outputs = model(images, csv_data)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            for i, oid in enumerate(orig_ids):
                if oid not in img_predictions:
                    img_predictions[oid] = []
                    img_labels[oid] = labels[i].item()
                img_predictions[oid].append(probs[i].item())

    y_true, y_pred = [], []
    correct = 0
    total = len(img_predictions)
    for oid, probs in img_predictions.items():
        pred = 1 if np.mean(probs) > 0.45 else 0
        true = img_labels[oid]
        y_true.append(true)
        y_pred.append(pred)
        if pred == true:
            correct += 1

    acc = 100 * correct / max(total, 1)
    return acc, y_true, y_pred

# ================= 5. 训练流程 =================
def run_experiment(mode_name, train_loader, val_loader, num_epochs=20, device='cuda', print_name=""):
    print(f"\n🔍 执行模型: {print_name}")
    model = RobustLateFusionNet(num_csv_features=13, num_classes=2, mode=mode_name).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.0003, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_yt, best_yp = [], []

    for epoch in range(num_epochs):
        model.train()
        for images, csv_data, labels, _ in train_loader:
            images, csv_data, labels = images.to(device), csv_data.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images, csv_data), labels)
            loss.backward()
            optimizer.step()

        acc, yt, yp = evaluate_image_level(model, val_loader, device)
        if acc > best_acc:
            best_acc = acc
            best_yt, best_yp = yt, yp

        if (epoch + 1) % 5 == 0 or epoch == 0:
            if mode_name == 'fusion':
                current_w = torch.sigmoid(model.w_csv).item()
                print(f"  Epoch [{epoch+1:2d}/{num_epochs}] | 大图准确率: {acc:.2f}%  实时动态 w_csv={current_w:.4f}")
            else:
                print(f"  Epoch [{epoch+1:2d}/{num_epochs}] | 大图准确率: {acc:.2f}%")
        scheduler.step()

    if mode_name == 'fusion':
        final_w = torch.sigmoid(model.w_csv).item()
        print(f"  -> 模型训练收敛，最终学习到的 CSV 动态信任权重 w_csv: {final_w:.4f}")

        # ==============================================================
        # 🌟 新增代码：找出验证集里真实的 False Positive 母图 🌟
        # ==============================================================
        model.eval()
        with torch.no_grad():
            for images, csv_data, labels, orig_ids in val_loader:
                images, csv_data = images.to(device), csv_data.to(device)
                outputs = model(images, csv_data)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                for i in range(len(orig_ids)):
                    pred = 1 if probs[i].item() > 0.45 else 0
                    true_label = labels[i].item()
                    # 预测为 1 (Banksy)，但真实标签为 0 (普通墙壁)
                    if pred == 1 and true_label == 0:
                        print(f"  ⚠️ 破案了！被 Fusion 模型误判为 Banksy 的普通墙壁母图是: 【 {orig_ids[i]} 】")
        # ==============================================================

    return best_acc, best_yt, best_yp

# ================= 6. 主程序 =================
if __name__ == '__main__':
    # 第一步：数据预处理（若已生成CSV可跳过）
    if os.path.exists(CSV_TRAIN) and os.path.exists(CSV_VAL):
        print("📁 检测到已有特征CSV，跳过切片步骤。")
        df_train = pd.read_csv(CSV_TRAIN)
        df_val = pd.read_csv(CSV_VAL)
    else:
        df_train, df_val = process_and_slice()

    # 数据增强与预处理
    transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    transform_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = V2Dataset(df_train, TILES_DIR, transform_train)
    val_dataset = V2Dataset(df_val, TILES_DIR, transform_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # 消融实验
    acc_img, yt_i, yp_i = run_experiment('image_only', train_loader, val_loader, 20, device,
                                         "Baseline 1: Vision Only (ResNet-18)")
    acc_csv, yt_c, yp_c = run_experiment('csv_only', train_loader, val_loader, 20, device,
                                         "Baseline 2: Attributes Only (13维手工特征)")
    acc_fus, yt_f, yp_f = run_experiment('fusion', train_loader, val_loader, 20, device,
                                         "Ours: Robust Late Fusion (可学习动态加权)")

    print("\n" + "="*50)
    print("🏆 真实运行得到的对比结果 🏆")
    print(f"  Vision Only (ResNet-18) : {acc_img:.2f}%")
    print(f"  CSV Only (13维统计特征) : {acc_csv:.2f}%")
    print(f"  Late Fusion (自适应加权) : {acc_fus:.2f}%")
    print(f"  📈 动态融合优势差距     : +{(acc_fus - acc_img):.2f}%")
    print("="*50)

    print("\n📊 最终大图级混淆矩阵 (供论文参考):")
    print("Vision Only:")
    print(confusion_matrix(yt_i, yp_i))
    print("Late Fusion:")
    print(confusion_matrix(yt_f, yp_f))

    # 可选：输出分类报告
    print("\n📋 Late Fusion 分类报告:")
    print(classification_report(yt_f, yp_f, target_names=['Negative (Wall)', 'Positive (Banksy)']))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')